# ColorSAR Model 1 — Full Dataset Training on Kaggle (2x Tesla T4 GPUs)
This notebook trains ColorSAR across 100% of the Sentinel-1 / Sentinel-2 dataset using **DataParallel** across both T4 GPUs.

In [ ]:
# 1. GPU Verification & Multi-GPU Detection
import os, torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

num_gpus = torch.cuda.device_count()
print(f"Detected GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  [GPU {i}]: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB VRAM)")

# Locate the dataset folder
dataset_root = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "s2" in dirs:
        dataset_root = os.path.dirname(root)
        break

if dataset_root:
    print(f"[*] Found dataset root: {dataset_root}")
else:
    print("[*] Looking in /kaggle/input:", os.listdir("/kaggle/input"))

In [ ]:
# 2. Model Architecture (Ensemble Encoder + U-Net Decoder + ColorizationModel)
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet50_Weights, DenseNet121_Weights

class EnsembleEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights_r = ResNet50_Weights.DEFAULT if pretrained else None
        weights_d = DenseNet121_Weights.DEFAULT if pretrained else None
        self.resnet50 = models.resnet50(weights=weights_r)
        self.densenet121 = models.densenet121(weights=weights_d)
        self.resnet50 = nn.Sequential(*list(self.resnet50.children())[:-2])
        self.densenet121.classifier = nn.Identity()

        self.conv1x1_resnet50 = nn.ModuleList([
            nn.Conv2d(256, 128, kernel_size=1), nn.Conv2d(512, 256, kernel_size=1),
            nn.Conv2d(1024, 512, kernel_size=1), nn.Conv2d(2048, 1024, kernel_size=1)
        ])
        self.conv1x1_densenet121 = nn.ModuleList([
            nn.Conv2d(256, 128, kernel_size=1), nn.Conv2d(512, 256, kernel_size=1),
            nn.Conv2d(1024, 512, kernel_size=1), nn.Conv2d(1024, 1024, kernel_size=1)
        ])
        self.fusion_blocks = nn.ModuleList([
            self.fusion_block(128, 128), self.fusion_block(256, 256),
            self.fusion_block(512, 512), self.fusion_block(1024, 1024)
        ])

    def fusion_block(self, r, d):
        return nn.Sequential(nn.Conv2d(r + d, r, kernel_size=1), nn.BatchNorm2d(r), nn.ReLU(inplace=True))

    def forward(self, x):
        resnet_features = []
        ri = x
        for i, layer in enumerate(self.resnet50.children()):
            ri = layer(ri)
            if i in [4, 5, 6, 7]:
                resnet_features.append(self.conv1x1_resnet50[i - 4](ri))

        densenet_features = []
        idx = 0
        di = x
        for i, layer in enumerate(self.densenet121.features):
            di = layer(di)
            if i in [4, 6, 8, 11]:
                densenet_features.append(self.conv1x1_densenet121[idx](di))
                idx += 1

        fused = []
        for i in range(4):
            f = torch.cat((resnet_features[i], densenet_features[i]), dim=1)
            fused.append(self.fusion_blocks[i](f))
        return fused

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decode1 = nn.Sequential(nn.Conv2d(1024, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(),
                                      nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False))
        self.decode2 = nn.Sequential(nn.Conv2d(1024, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(),
                                      nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False))
        self.decode3 = nn.Sequential(nn.Conv2d(512, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(),
                                      nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False))
        self.decode4 = nn.Sequential(nn.Conv2d(256, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
                                      nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False))
        self.decode5 = nn.Sequential(nn.Conv2d(64, 2, 3, 1, 1), nn.BatchNorm2d(2), nn.Tanh(),
                                      nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False))

    def forward(self, f7, f14, f28, f56):
        x = self.decode1(f7)
        x = torch.cat([x, f14], dim=1)
        x = self.decode2(x)
        x = torch.cat([x, f28], dim=1)
        x = self.decode3(x)
        x = torch.cat([x, f56], dim=1)
        x = self.decode4(x)
        return self.decode5(x)

class ColorizationModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        f56, f28, f14, f7 = self.encoder(x)
        return self.decoder(f7, f14, f28, f56)

print("Model architecture defined successfully.")

In [ ]:
# 3. Data Pipeline & Dataset Class
import cv2
import numpy as np
from PIL import Image
from skimage.color import rgb2lab, lab2rgb
from torch.utils.data import Dataset, DataLoader, random_split

def find_optical_images(root_dir):
    paths = []
    for root, dirs, files in os.walk(root_dir):
        if os.path.basename(root) == "s2":
            for f in sorted(files):
                if f.endswith(".png") or f.endswith(".jpg") or f.endswith(".jpeg"):
                    paths.append(os.path.join(root, f))
    return sorted(paths)

def rgb_to_lab(img):
    lab = rgb2lab(img).astype("float32")
    lab = np.transpose(lab, (2, 0, 1))
    L = 2 * lab[[0], ...] / 100 - 1
    ab = 2 * (lab[[1, 2], ...] + 128) / 255 - 1
    return L, ab

class ColorizationDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.paths[idx]).convert("RGB"))
        img = cv2.resize(img, (224, 224))
        L, ab = rgb_to_lab(img)
        return torch.from_numpy(L), torch.from_numpy(ab)

print("Data pipeline defined.")

In [ ]:
# 4. Full Dataset Training on 2x Tesla T4 GPUs
import time, json
import torch.optim as optim
import matplotlib.pyplot as plt

# =========================================================================
# OPTIMAL HYPERPARAMETERS FOR ENTIRE DATASET & 2x T4 GPUs
# =========================================================================
N_IMAGES = None       # None = 100% of all images in dataset (~16,000 pairs)
EPOCHS = 20           # 20 epochs gives excellent convergence (~25-30 min total)
BATCH_SIZE = 64       # 64 total (32 per GPU across both T4 GPUs)
LEARNING_RATE = 5e-4  # 0.0005 with Adam is optimal for batch size 64
NUM_WORKERS = 4       # Uses all 4 vCPUs on Kaggle for zero bottleneck
OUTPUT_DIR = "/kaggle/working"
# =========================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Collect all optical images
all_paths = find_optical_images(dataset_root if dataset_root else "/kaggle/input")
print(f"[*] Total optical images found across all terrains: {len(all_paths)}")
assert len(all_paths) > 0, "No images found! Verify dataset is added."

if N_IMAGES is not None and N_IMAGES < len(all_paths):
    rng = np.random.RandomState(42)
    idx = rng.permutation(len(all_paths))[:N_IMAGES]
    paths = [all_paths[i] for i in idx]
    print(f"[*] Using subset of {len(paths)} images")
else:
    paths = all_paths
    print(f"[*] Training on FULL DATASET: {len(paths)} images (100%)")

dataset = ColorizationDataset(paths)
n_val = max(int(0.10 * len(dataset)), BATCH_SIZE)
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=NUM_WORKERS, pin_memory=True)
print(f"[*] Training set: {len(train_ds)} samples ({len(train_loader)} batches/epoch)")
print(f"[*] Validation set: {len(val_ds)} samples ({len(val_loader)} batches/epoch)")

# Instantiate raw models
raw_encoder = EnsembleEncoder(pretrained=True)
raw_encoder.eval()
for p in raw_encoder.parameters():
    p.requires_grad = False  # Keep encoder frozen

raw_decoder = Decoder()
base_model = ColorizationModel(raw_encoder, raw_decoder).to(device)

# Wrap entire model in DataParallel for seamless 2x GPU scaling without PCIe bottleneck
if torch.cuda.device_count() > 1:
    print(f"[*] Activating nn.DataParallel across {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(base_model)
else:
    model = base_model

criterion = nn.MSELoss()
# Only optimize decoder parameters
decoder_params = base_model.decoder.parameters()
optimizer = optim.Adam(decoder_params, lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

history = {"train_loss": [], "val_loss": []}
best_val = float("inf")
t0 = time.time()

print("\n" + "="*60)
print(f"  STARTING TRAINING — {EPOCHS} EPOCHS ACROSS {torch.cuda.device_count()} GPU(S)")
print("="*60)

for epoch in range(EPOCHS):
    t_ep = time.time()
    # --- Training Phase ---
    base_model.decoder.train()
    base_model.encoder.eval()
    total_train = 0.0
    for L, ab in train_loader:
        L, ab = L.to(device, non_blocking=True), ab.to(device, non_blocking=True)
        L_3ch = L.repeat(1, 3, 1, 1)
        out = model(L_3ch)
        loss = criterion(out, ab)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train += loss.item()
    
    tr_loss = total_train / max(len(train_loader), 1)
    scheduler.step()

    # --- Validation Phase ---
    base_model.decoder.eval()
    total_val = 0.0
    with torch.no_grad():
        for L, ab in val_loader:
            L, ab = L.to(device, non_blocking=True), ab.to(device, non_blocking=True)
            L_3ch = L.repeat(1, 3, 1, 1)
            out = model(L_3ch)
            total_val += criterion(out, ab).item()
    va_loss = total_val / max(len(val_loader), 1)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    ep_sec = time.time() - t_ep
    total_min = (time.time() - t0) / 60
    print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] ({ep_sec:.1f}s) -> Train MSE: {tr_loss:.4f} | Val MSE: {va_loss:.4f} | Total: {total_min:.1f}m")

    # Save best checkpoint (saves pure decoder state dict for local reuse)
    if va_loss < best_val:
        best_val = va_loss
        torch.save(base_model.decoder.state_dict(), os.path.join(OUTPUT_DIR, "model_1.pth"))
        print(f"   --> [*] Saved new best model checkpoint to {OUTPUT_DIR}/model_1.pth")

# Save training history json and loss curve
json.dump(history, open(os.path.join(OUTPUT_DIR, "history.json"), "w"), indent=2)
plt.figure(figsize=(8, 4.5))
plt.plot(range(1, EPOCHS + 1), history["train_loss"], marker="o", label="Train Loss (MSE)", color="#1f77b4")
plt.plot(range(1, EPOCHS + 1), history["val_loss"], marker="s", label="Val Loss (MSE)", color="#ff7f0e")
plt.xlabel("Epoch", fontsize=12); plt.ylabel("Loss (MSE)", fontsize=12)
plt.title("ColorSAR Model 1 — Training & Validation Loss Curve", fontsize=13)
plt.grid(True, linestyle="--", alpha=0.6); plt.legend(fontsize=11); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curve.png"), dpi=150)
plt.show()
print("\n[*] TRAINING COMPLETE! Saved model_1.pth, loss_curve.png, and history.json")

In [ ]:
# 5. Generate and Display Visual Predictions on Validation Data
base_model.decoder.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "model_1.pth"), map_location=device))
base_model.eval()

L_batch, ab_batch = next(iter(val_loader))
n = min(4, len(L_batch))
L_n, ab_n = L_batch[:n].to(device), ab_batch[:n].to(device)

with torch.no_grad():
    pred_n = base_model(L_n.repeat(1, 3, 1, 1))

def lab_tensor_to_rgb(L_t, ab_t):
    L_u = (L_t.cpu().numpy() + 1) * 0.5 * 100
    ab_u = (ab_t.cpu().numpy() + 1) * 0.5 * 255 - 128
    lab = np.transpose(np.concatenate([L_u, ab_u], axis=0), (1, 2, 0))
    return lab2rgb(lab)

fig, axes = plt.subplots(3, n, figsize=(3.5 * n, 9.5))
for i in range(n):
    axes[0, i].imshow(L_n[i, 0].cpu().numpy(), cmap="gray")
    axes[0, i].set_title(f"Input L (Sample {i+1})", fontsize=11)
    axes[1, i].imshow(lab_tensor_to_rgb(L_n[i], ab_n[i]))
    axes[1, i].set_title(f"Ground Truth RGB", fontsize=11)
    axes[2, i].imshow(lab_tensor_to_rgb(L_n[i], pred_n[i]))
    axes[2, i].set_title(f"ColorSAR Output", fontsize=11)
    for r in range(3): axes[r, i].axis("off")
plt.suptitle("ColorSAR Model 1: Ground Truth vs Model Colorization", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_predictions.png"), dpi=150, bbox_inches="tight")
plt.show()
print("[*] Sample visual comparison generated: sample_predictions.png")